## CS3570 Introduction to Multimedia Technology
### Homework 04 Due: 11:59pm, 2026/05/25

### Reminder:
* **You are not allowed to use any function that directly generates the result for each step, except for those provided by us.**
* **Your code must display and output your results to enable us to verify its correctness.**
* **Please follow the instructions in the Jupyter Notebook and complete the parts marked as `"TODO."`**
* **Please include your report in the Jupyter Notebook using markdown.**
* **Please compress your code, training script, and result images in a zip file named HW4_{StudentID}.zip and upload it to eeclass. Do NOT include the dataset or trained checkpoints.**
* **Homework should be submitted before the announced due time. Scores of late homework will be reduced by 20% per day.**
* **If you encounter any problems or have questions, please post them on eeclass.**

### Topic: Image Restoration on JPEG + Noisy Inputs
You have learned **JPEG compression** in this course. Aggressive quantization gives high compression ratios but introduces visible **artifacts** — blockiness, ringing, color bleeding. Real-world images also pick up **sensor / channel noise**. In this homework you train neural networks that take an image corrupted by **heavy JPEG (qf ∈ [5, 20]) plus additive Gaussian noise (σ ∈ [0, 0.1] in [-1, 1] units)** and output something as close as possible to the clean original.

You will implement **two** restoration architectures and run an **ablation study** on each:

- **Part 1 (45%): Naive CNN** with residual blocks
- **Part 2 (45%): UNet** with multi-scale skip connections
- **Report (10%)**

Within each part you will train **one base configuration** plus **three single-axis ablations** (size / loss / one other hyperparameter) plus **one final "best" run** chosen from your observations — five runs per architecture, ten runs total. The minimum compute is `5 × 2000 = 10K` steps per architecture.

The losses are based on [SenseCraft](https://github.com/KohakuBlueleaf/SenseCraft) — a small library that lets you mix MSE / L1 / SSIM / LPIPS / ... by passing a list of dicts. The default mix is `mse + 0.1 * ssim`. The loss-ablation section asks you to compare alternatives — see the SenseCraft README for the available terms.

In [ ]:
# Local setup: from the repo root, `pip install -r 04-neural-image-restoration/requirements.txt`.
# On Colab the next line is enough:
%pip install lightning anyschedule sensecraft ipywidgets huggingface_hub -q

In [ ]:
# Imports
import io
import os
import math
import random
from IPython.display import display
from ipywidgets import Output

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as data
from torchvision.transforms import transforms
from anyschedule import AnySchedule
from sensecraft.loss import SenseCraftLoss
from sensecraft.metrics import psnr, ssim
from PIL import Image
from tqdm.auto import tqdm

import lightning.pytorch as pl
from lightning.pytorch import LightningModule, Trainer
from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor

import matplotlib.pyplot as plt

if not os.path.exists("output"):
    os.makedirs("output")

In [ ]:
# Helpers
def denormalize(x):
    """[-1, 1] -> [0, 1]"""
    return x * 0.5 + 0.5


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def display_sample(image, widget):
    with widget:
        display(image, clear=True)


def _grid_image(t, nrow):
    """t: [B, 3, H, W] in [-1, 1] -> single PIL image, `nrow` images per row."""
    t = denormalize(t).clamp(0, 1)
    B, C, H, W = t.shape
    nrows = (B + nrow - 1) // nrow
    canvas = torch.zeros(C, nrows * H, nrow * W, device=t.device)
    for i in range(B):
        r, c = i // nrow, i % nrow
        canvas[:, r * H:(r + 1) * H, c * W:(c + 1) * W] = t[i]
    arr = (canvas * 255).cpu().numpy().astype(np.uint8).transpose(1, 2, 0)
    return Image.fromarray(arr)


# All tensors live in [-1, 1], so PSNR / SSIM use data_range=2.0.
DATA_RANGE = 2.0

In [ ]:
# Device selection
try:
    import torch_xla
    HAS_TPU = True
except ImportError:
    HAS_TPU = False

if HAS_TPU:
    ACCELERATOR, PRECISION = "tpu", "bf16-mixed"
elif torch.cuda.is_available():
    ACCELERATOR, PRECISION = "gpu", "16-mixed"
else:
    ACCELERATOR, PRECISION = "cpu", "32-true"

print(f"accelerator: {ACCELERATOR}, precision: {PRECISION}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

### Dataset & Degradation — Image Restoration on JPEG + Noisy Inputs

We work at **256x256** resolution. The training data is a class-balanced subset of
**ImageNet-1k** — 50 train + 10 val images per class (50K + 10K total) — pre-resized
to a short side of 256 and stored as **lossless WebP** so the clean targets are not
themselves contaminated with JPEG artifacts.

Each `(degraded, clean)` pair is built on the fly by applying **two compounding
degradations** in this order:

1. take a clean RGB image (already short-side=256), random/center crop to 256x256
2. **JPEG round-trip** through PIL at a quality factor sampled uniformly from
   `QF_RANGE = (5, 20)` — heavy compression with visible blocking and ringing
3. **Additive Gaussian noise** with standard deviation sampled uniformly from
   `NOISE_RANGE = (0.0, 0.1)` (in the `[-1, 1]` tensor scale, ~ `[0, 13]` in 8-bit
   pixel units), then clamp back to `[-1, 1]`
4. normalize the clean target to `[-1, 1]`

Validation uses the midpoints of both ranges (`qf=12`, `sigma≈0.05`) so the val
PSNR is a deterministic, repeatable measurement.

The cell below downloads `train.tar` and `val.tar` and extracts them to `./data/`.
Re-running is a no-op once the data is present.

In [ ]:
# Download and extract the prepared dataset from HuggingFace (~5 GB total).
# The tar files live at https://huggingface.co/datasets/KBlueLeaf/inet-sub
# and are cached under ~/.cache/huggingface — re-running is a no-op.
import os, tarfile
from huggingface_hub import hf_hub_download

REPO = "KBlueLeaf/inet-sub"

def _fetch(filename, extract_to):
    if os.path.isdir(extract_to):
        print(f"[skip] {extract_to} already exists")
        return
    tar_path = hf_hub_download(repo_id=REPO, filename=filename, repo_type="dataset")
    print(f"[extract] {tar_path} -> {extract_to}")
    os.makedirs(extract_to, exist_ok=True)
    with tarfile.open(tar_path) as tf:
        tf.extractall(extract_to)

os.makedirs("data", exist_ok=True)
_fetch("train.tar", "data/train")
_fetch("val.tar",   "data/val")
print("train classes:", len(os.listdir("data/train")),
      "val classes:",   len(os.listdir("data/val")))

In [ ]:
from torchvision import datasets


def jpeg_degrade_pil(pil_img, quality):
    """Round-trip a PIL RGB image through JPEG at the given quality. Returns PIL."""
    buf = io.BytesIO()
    pil_img.save(buf, format="JPEG", quality=int(quality))
    buf.seek(0)
    return Image.open(buf).convert("RGB").copy()


class JPEGRestorationDataset(data.Dataset):
    """Wrap an ImageFolder of pre-resized clean images into (degraded, clean) tensors in [-1, 1].

    Degradation pipeline per sample:
      clean -> JPEG-encode/decode at qf ~ U(qf_range)
            -> add Gaussian noise with sigma ~ U(noise_range), in [-1, 1] units
            -> clamp to [-1, 1]
    Validation uses the midpoints of both ranges so the eval signal is deterministic.
    """

    def __init__(self, root, img_size=256, qf_range=(5, 20),
                 noise_range=(0.0, 0.1), train=True):
        self.source = datasets.ImageFolder(root)
        self.img_size = img_size
        self.qf_range = qf_range
        self.noise_range = noise_range
        self.train = train
        if train:
            self.spatial = transforms.Compose([
                transforms.RandomCrop(img_size, pad_if_needed=True),
                transforms.RandomHorizontalFlip(),
            ])
        else:
            self.spatial = transforms.CenterCrop(img_size)
        self.to_tensor = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
        ])

    def __len__(self):
        return len(self.source)

    def __getitem__(self, idx):
        clean_pil, _ = self.source[idx]
        if clean_pil.mode != "RGB":
            clean_pil = clean_pil.convert("RGB")
        clean_pil = self.spatial(clean_pil)
        if self.train:
            qf = random.randint(self.qf_range[0], self.qf_range[1])
            sigma = random.uniform(self.noise_range[0], self.noise_range[1])
        else:
            qf = (self.qf_range[0] + self.qf_range[1]) // 2
            sigma = 0.5 * (self.noise_range[0] + self.noise_range[1])
        degraded_pil = jpeg_degrade_pil(clean_pil, qf)
        clean = self.to_tensor(clean_pil)
        deg = self.to_tensor(degraded_pil)
        if sigma > 0:
            deg = (deg + torch.randn_like(deg) * sigma).clamp(-1, 1)
        return deg, clean


def build_jpeg_datasets(img_size=256, qf_range=(5, 20),
                        noise_range=(0.0, 0.1), root="./data"):
    return (
        JPEGRestorationDataset(f"{root}/train", img_size, qf_range, noise_range, train=True),
        JPEGRestorationDataset(f"{root}/val",   img_size, qf_range, noise_range, train=False),
    )


# Constants used by every part below
IMG_SIZE = 256
QF_RANGE = (5, 20)
NOISE_RANGE = (0.0, 0.1)

In [ ]:
# Visualize a few (degraded, clean) pairs and report input PSNR / SSIM
train_ds, val_ds = build_jpeg_datasets(img_size=IMG_SIZE, qf_range=QF_RANGE, noise_range=NOISE_RANGE)
print(f"train: {len(train_ds)}, val: {len(val_ds)}")

n = 4
mid_qf = (QF_RANGE[0] + QF_RANGE[1]) // 2
mid_sigma = 0.5 * (NOISE_RANGE[0] + NOISE_RANGE[1])
fig, axes = plt.subplots(2, n, figsize=(3 * n, 6))
for i in range(n):
    deg, clean = val_ds[i]
    axes[0, i].imshow(denormalize(deg).permute(1, 2, 0).numpy().clip(0, 1))
    axes[0, i].set_title(f"Degraded (qf={mid_qf}, σ={mid_sigma:.2f})")
    axes[0, i].axis("off")
    axes[1, i].imshow(denormalize(clean).permute(1, 2, 0).numpy().clip(0, 1))
    axes[1, i].set_title("Clean")
    axes[1, i].axis("off")
plt.tight_layout()
plt.savefig("output/dataset_preview.png", dpi=120)
plt.show()

deg, clean = val_ds[0]
deg_b, clean_b = deg.unsqueeze(0), clean.unsqueeze(0)
print(f"Input PSNR (degraded vs. clean, qf={mid_qf}, σ={mid_sigma:.2f}): "
      f"{psnr(deg_b, clean_b, data_range=DATA_RANGE).item():.2f} dB | "
      f"SSIM: {ssim(deg_b, clean_b, data_range=DATA_RANGE).item():.4f}")

### Shared training scaffolding (used by Part 1 / Part 2)

Same loss, same optimizer, same training loop — only the model changes.

The default loss mix is `mse + 0.1 * ssim`. You can pass any `SenseCraftLoss` config — see the [SenseCraft README](https://github.com/KohakuBlueleaf/SenseCraft) for the full list of supported terms (`l1`, `mse`, `ssim`, `ms_ssim`, `lpips`, `dists`, ...). Adding a perceptual term such as `lpips` typically improves visual sharpness at a small PSNR cost.

In [ ]:
restoration_display = Output()


class RestorationModule(LightningModule):
    """Generic restoration trainer (used for Part 1 and Part 2)."""

    def __init__(
        self,
        model: nn.Module,
        loss_cfg=None,
        lr=2e-4,
        total_steps=1e6,
        warmup_steps=500,
        weight_decay=1e-3,
        log_period=100,
        name="restoration",
    ):
        super().__init__()
        self.model = model
        if loss_cfg is None:
            loss_cfg = [{"mse": 1.0}, {"ssim": 0.1}]
        self.loss = SenseCraftLoss(loss_cfg, return_dict=False)
        self.lr = lr
        self.total_steps = int(total_steps)
        self.warmup_steps = warmup_steps
        self.weight_decay = weight_decay
        self.log_period = log_period
        self.name = name
        self._preview_batch = None

    def configure_optimizers(self):
        opt = optim.AdamW(
            self.model.parameters(), lr=self.lr,
            weight_decay=self.weight_decay, betas=(0.9, 0.99), eps=1e-6,
        )
        sched = AnySchedule(opt, config={"lr": {
            "mode": "cosine", "end": self.total_steps,
            "min_value": 0.01, "warmup": self.warmup_steps,
        }})
        return {"optimizer": opt, "lr_scheduler": {"scheduler": sched, "interval": "step"}}

    def forward(self, x):
        return self.model(x)

    @torch.no_grad()
    def draw_sample(self, deg, clean, n=4):
        self.model.eval()
        deg = deg[:n].to(self.device)
        clean = clean[:n].to(self.device)
        pred = self.model(deg).clamp(-1, 1)
        triplet = torch.cat([deg, pred, clean], dim=0)
        img = _grid_image(triplet, nrow=n)
        display_sample(img, restoration_display)
        img.save(f"output/{self.name}-sample.png")
        self.model.train()

    def training_step(self, batch, batch_idx):
        deg, clean = batch
        if self._preview_batch is None:
            self._preview_batch = (deg.detach().clone(), clean.detach().clone())
        pred = self.model(deg)
        loss = self.loss(pred, clean)
        self.log("loss", loss, prog_bar=True)
        with torch.no_grad():
            self.log("psnr", psnr(pred.clamp(-1, 1), clean, data_range=DATA_RANGE), prog_bar=True)
        if self.global_step % self.log_period == 0:
            d, c = self._preview_batch
            self.draw_sample(d, c)
        return loss

    def validation_step(self, batch, batch_idx):
        deg, clean = batch
        pred = self.model(deg).clamp(-1, 1)
        self.log("val_loss", self.loss(pred, clean), prog_bar=True, sync_dist=True)
        self.log("val_psnr", psnr(pred, clean, data_range=DATA_RANGE), prog_bar=True, sync_dist=True)
        self.log("val_ssim", ssim(pred, clean, data_range=DATA_RANGE), prog_bar=True, sync_dist=True)


def make_loaders(train_batch_size=16, val_batch_size=64, num_workers=4):
    """Train / val DataLoaders. Val uses a larger batch size since no backward pass is needed."""
    train_ds, val_ds = build_jpeg_datasets(
        img_size=IMG_SIZE, qf_range=QF_RANGE, noise_range=NOISE_RANGE,
    )
    nw = 0 if ACCELERATOR == "tpu" else num_workers
    train_loader = data.DataLoader(
        train_ds, batch_size=train_batch_size, shuffle=True,
        num_workers=nw, persistent_workers=nw > 0, drop_last=True,
    )
    val_loader = data.DataLoader(
        val_ds, batch_size=val_batch_size, shuffle=False,
        num_workers=nw, persistent_workers=nw > 0,
    )
    return train_loader, val_loader


def final_eval(module, val_loader):
    """Full val pass via Lightning's trainer.validate. Returns (psnr_dB, ssim).

    Spawns a fresh Trainer with the same accelerator + precision as training so:
      - Lightning re-places the module on the accelerator (forward runs on GPU, not CPU)
      - forward uses the same mixed-precision config as training
      - val_psnr and val_ssim are aggregated across the FULL val_loader
    """
    eval_trainer = Trainer(
        accelerator=ACCELERATOR, precision=PRECISION,
        logger=False, enable_model_summary=False,
    )
    metrics = eval_trainer.validate(module, val_loader, verbose=False)
    return float(metrics[0]["val_psnr"]), float(metrics[0]["val_ssim"])

In [ ]:
# Live preview widget — run this once. Every Part will draw rolling samples here.
display(restoration_display)

## Part 1: Naive CNN with Residual Blocks (45%)

Build a simple but non-trivial CNN: a stack of **residual blocks** that maps the degraded image directly to the clean image.

### Design rules

**Mandatory** (your implementation must satisfy these):
- a stack of **residual blocks** of the form `y = x + f(x)`
- spatial dimensions preserved end-to-end (`H_out == H_in`, `W_out == W_in`)

**Free to explore.** The `# TODO` skeleton below is *one* valid implementation, not the only one. You're encouraged to deviate — for example:
- swap the activation, add normalization, or use bottleneck blocks
- mix kernel sizes between blocks
- try grouped / depthwise / dilated convolutions
- tweak the initialization scheme
- add channel attention or other lightweight refinements

…as long as the mandatory rules above still hold. A creative-but-correct implementation that beats the skeleton on val PSNR is fully acceptable.

### Ablation study

You will perform a **5-run ablation study** on whatever architecture you implement:

| Sub-section | What you do | Points |
|---|---|---|
| **1(a)** Implementation + Base run | Implement `ResBlock` and `NaiveCNN`; train one **base** configuration | 10% |
| **1(b)** Size ablation | Re-train with a **different size** (`hidden_dim` / `num_blocks` / `kernel_size`), all else identical | 10% |
| **1(c)** Loss ablation | Re-train with a **different `LOSS_CFG`**, all else identical | 10% |
| **1(d)** Other-hyperparameter ablation | **You pick** one hyperparameter (`lr`, `weight_decay`, `warmup_steps`, `batch_size`, `max_steps`, ...) and try at least 2 values | 10% |
| **1(e)** Best configuration | Combine the lessons above into the configuration **you believe will perform best**, and train it | 5% |

**How to actually do this in the notebook:** there is one shared training cell and one shared eval cell below. Re-run them with edited config for each ablation, and **set `RUN_NAME` differently each time** (`"base"`, `"size_X"`, `"loss_X"`, `"hp_X"`, `"best"`) so the saved sample images don't overwrite each other. Record `(RUN_NAME, params, val_psnr)` in the results cell at the end of Part 1.

In [ ]:
# TODO: Implement a residual block: y = x + W2(act(W1(x))).
#       Use SiLU or Mish (NOT ReLU/GELU). Padding must keep H, W unchanged.
class ResBlock(nn.Module):
    def __init__(self, channels, kernel_size=3):
        super().__init__()
        pad = kernel_size // 2  # same-padding for odd kernels keeps H, W unchanged
        self.conv1 = nn.Conv2d(channels, channels, kernel_size, padding=pad)
        self.act = nn.SiLU()
        self.conv2 = nn.Conv2d(channels, channels, kernel_size, padding=pad)

    def forward(self, x):
        return x + self.conv2(self.act(self.conv1(x)))


# TODO: Implement a residual CNN restoration model.
#       Architecture: conv_in -> [ResBlock] x num_blocks -> conv_out
class NaiveCNN(nn.Module):
    def __init__(self, in_channels=3, hidden_dim=32, num_blocks=6, kernel_size=3):
        super().__init__()
        pad = kernel_size // 2
        self.conv_in = nn.Conv2d(in_channels, hidden_dim, kernel_size, padding=pad)
        self.blocks = nn.Sequential(*[
            ResBlock(hidden_dim, kernel_size=kernel_size) for _ in range(num_blocks)
        ])
        self.conv_out = nn.Conv2d(hidden_dim, in_channels, kernel_size, padding=pad)

    def forward(self, x):
        x = self.conv_in(x)
        x = self.blocks(x)
        return self.conv_out(x)


# Sanity check (will only succeed once your implementation is complete)
try:
    _m = NaiveCNN(hidden_dim=32, num_blocks=6)
    print("output:", _m(torch.randn(2, 3, IMG_SIZE, IMG_SIZE)).shape, "params:", count_params(_m))
except Exception as e:
    print("ResBlock / NaiveCNN not implemented yet:", e)

### 1(a) Implementation + Base Run (10%)

The cell below trains the **base** configuration. For ablations 1(b) / (c) / (d) and the final best run 1(e), come back to this cell, edit `RUN_NAME` and the relevant hyperparameter(s), and re-run both this cell and the eval cell that follows.

In [ ]:
pl.seed_everything(42)

# === Re-run this cell for each ablation. Change RUN_NAME so saved samples don't collide. ===
RUN_NAME = "base"   # "base" | "size_<tag>" | "loss_<tag>" | "hp_<tag>" | "best"

# --- Compute budget (kept identical across all 5 runs unless you ablate it in 1(d)) ---
MAX_STEPS = 2000
VAL_EVERY = 500
VAL_BATCHES = 50
TRAIN_BATCH_SIZE = 16
VAL_BATCH_SIZE = 64    # larger batch for val — no backward, more throughput

# --- Size ---
HIDDEN_DIM = 32
NUM_BLOCKS = 6
KERNEL_SIZE = 3

# --- Optimizer / schedule ---
LR = 2e-4
WARMUP_STEPS = 200
WEIGHT_DECAY = 1e-3

# --- Loss ---
LOSS_CFG = [{"mse": 1.0}, {"ssim": 0.1}]

train_loader, val_loader = make_loaders(TRAIN_BATCH_SIZE, VAL_BATCH_SIZE)

p1_model = NaiveCNN(
    in_channels=3, hidden_dim=HIDDEN_DIM,
    num_blocks=NUM_BLOCKS, kernel_size=KERNEL_SIZE,
)
print(f"[{RUN_NAME}] Part 1 NaiveCNN params: {count_params(p1_model)/1e6:.3f} M")

p1 = RestorationModule(
    model=p1_model, loss_cfg=LOSS_CFG, lr=LR,
    total_steps=MAX_STEPS, warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY, log_period=100,
    name=f"p1-naivecnn-{RUN_NAME}",
)

trainer = Trainer(
    max_steps=MAX_STEPS, max_epochs=-1,
    accelerator=ACCELERATOR, precision=PRECISION,
    callbacks=[LearningRateMonitor(logging_interval="step")],
    log_every_n_steps=10, gradient_clip_val=1.0,
    val_check_interval=VAL_EVERY, limit_val_batches=VAL_BATCHES,
)

trainer.fit(p1, train_loader, val_loader)

In [ ]:
# === Part 1 — Final evaluation for the run named RUN_NAME ===
# Uses Lightning's trainer.validate, so forward runs on the accelerator under the
# trainer's mixed-precision config (same as training-time val).
val_psnr, val_ssim = final_eval(p1, val_loader)
params_M = count_params(p1.model) / 1e6
print(f"Part 1 NaiveCNN [{RUN_NAME}]  |  params: {params_M:.3f} M  "
      f"|  val PSNR: {val_psnr:.3f} dB  |  val SSIM: {val_ssim:.4f}")

deg, clean = next(iter(val_loader))
p1.draw_sample(deg, clean, n=4)
print(f"saved: output/p1-naivecnn-{RUN_NAME}-sample.png "
      f"— inspect this for the qualitative discussion in your report.")

### 1(b) Size Ablation (10%)

**Goal:** observe how the model's parameter count affects val PSNR at fixed compute.

Re-run the training cell above with **at least one alternative size configuration** — change one or more of `HIDDEN_DIM`, `NUM_BLOCKS`, `KERNEL_SIZE`. Keep `LOSS_CFG` and the optimizer / schedule **identical to the base run**. Set `RUN_NAME = "size_<tag>"` (e.g. `"size_h64_b8"`) so the saved sample doesn't overwrite the base sample.

After running, fill in the new row in the results cell at the end of Part 1 and write 1–2 sentences summarising what you observed.

### 1(c) Loss Ablation (10%)

**Goal:** observe how the choice of loss function affects val PSNR and qualitative output at fixed model size and compute.

Re-run the training cell with **at least one alternative `LOSS_CFG`**. Suggestions from the [SenseCraft README](https://github.com/KohakuBlueleaf/SenseCraft):

- pure `[{"mse": 1.0}]`
- `[{"l1": 1.0}, {"ssim": 0.1}]`
- `[{"mse": 1.0}, {"lpips": 0.05}]`
- `[{"l1": 1.0}, {"ms_ssim": 0.1}]`

Keep size and the rest of the hyperparameters **identical to the base run**. Set `RUN_NAME = "loss_<tag>"` (e.g. `"loss_pure_mse"`).

In your results discussion, comment on **both** the PSNR change and the visual difference between samples — pixel losses (MSE, L1) and perceptual / structural losses (SSIM, LPIPS) usually trade off against each other.

### 1(d) Hyperparameter Ablation (10%)

**Goal:** convince yourself that *some* other hyperparameter actually matters (or doesn't).

**Pick one** hyperparameter from the list below and re-train with **at least one alternative value** while keeping size, loss, and the rest fixed. Set `RUN_NAME = "hp_<param>_<value>"` (e.g. `"hp_lr_5e-4"`).

Reasonable choices:

- `LR` (e.g. `5e-4`, `1e-4`)
- `WEIGHT_DECAY` (e.g. `0`, `1e-2`)
- `WARMUP_STEPS` (e.g. `0`, `500`)
- `TRAIN_BATCH_SIZE` (e.g. `8`, `32` — note this changes the effective LR/step budget)
- `MAX_STEPS` (e.g. `1000`, `4000` — note this also changes the cosine schedule)

Discuss in your results section **why** the change helped / hurt — knowing the *direction* and *magnitude* of an effect is the actual research skill being tested here.

### 1(e) Best Configuration (5%)

Based on what you observed in 1(b)/(c)/(d), pick what you think is **the best configuration** within the same compute budget (`MAX_STEPS = 2000`) and train one final run. Set `RUN_NAME = "best"`. In your results discussion, **justify in 2–3 sentences** which observations from the ablations led to this choice.

This is a deliberate hyperparameter-tuning exercise — there is no single right answer. The grading rewards a reasonable choice + an honest justification.

### Part 1 Results

After running all 5 configurations (base + 3 ablations + best), fill in the table in the cell below and run it. The cell will print a summary and save a comparison plot to `output/p1_results.png`.

In [ ]:
# Part 1 results — fill this in after running all 5 configurations.
# Each entry: (RUN_NAME, params_M, val_psnr_dB, val_ssim, brief_note)
p1_results = [
    # ("base",         0.20, 28.5, 0.85, "h32 b6 k3, mse+0.1*ssim"),
    # ("size_h64_b8",  0.95, 28.9, 0.86, "bigger model"),
    # ("loss_l1ssim",  0.20, 28.7, 0.87, "L1 + 0.1*SSIM"),
    # ("hp_lr_5e-4",   0.20, 28.4, 0.84, "higher LR"),
    # ("best",         0.95, 29.1, 0.88, "h64 b8 + L1+SSIM"),
]

if p1_results:
    labels = [r[0] for r in p1_results]
    psnrs  = [r[2] for r in p1_results]
    ssims  = [r[3] for r in p1_results]

    print(f"{'run':<22} {'params (M)':>11} {'PSNR (dB)':>11} {'SSIM':>8}  note")
    print("-" * 80)
    for r in p1_results:
        print(f"{r[0]:<22} {r[1]:>11.3f} {r[2]:>11.3f} {r[3]:>8.4f}  {r[4]}")

    fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 4))
    for ax, vals, ylabel, color in [
        (axL, psnrs, "val PSNR (dB)", "steelblue"),
        (axR, ssims, "val SSIM",      "seagreen"),
    ]:
        bars = ax.bar(labels, vals, color=color)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, v, f"{v:.3f}",
                    ha="center", va="bottom", fontsize=9)
        ax.set_ylabel(ylabel)
        ax.set_ylim(min(vals) - 0.5 * (max(vals) - min(vals) + 1e-3),
                    max(vals) + 0.5 * (max(vals) - min(vals) + 1e-3))
        for tick in ax.get_xticklabels():
            tick.set_rotation(20)
            tick.set_horizontalalignment("right")
    fig.suptitle("Part 1 — NaiveCNN ablation results")
    plt.tight_layout()
    plt.savefig("output/p1_results.png", dpi=120)
    plt.show()
else:
    print("Fill in p1_results above after running all 5 configurations.")

### Qualitative comparison — base vs ablation (error heatmaps)

The cell below builds a **side-by-side figure + per-pixel absolute error heatmaps** from the saved `output/p1-naivecnn-*-sample.png` grids (same layout as `draw_sample`: row 1 = degraded, row 2 = prediction, row 3 = ground truth).

**Why error heatmaps?** Restoration papers routinely visualize *where* a model still disagrees with the target as a map of \(|\hat{I} - I|\) (often averaged over RGB). Brighter regions = larger pixel error (blocking, ringing, leftover noise).

**Sources (official docs / standard practice):**
- Matplotlib `imshow` for pseudocolor maps: [Image annotated heatmap gallery](https://matplotlib.org/stable/gallery/images_contours_and_fields/image_annotated_heatmap.html) and [`Axes.imshow` API](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.imshow.html)
- NumPy element-wise absolute difference: [`numpy.abs`](https://numpy.org/doc/stable/reference/generated/numpy.abs.html)
- Error / residual maps in image restoration (example usage in a widely cited denoising paper): [Zhang et al., *Beyond a Gaussian Denoiser* (DnCNN), arXiv:1608.03981](https://arxiv.org/abs/1608.03981) — qualitative figures compare outputs and highlight remaining structure in the residual.

Run after you have saved at least `base` and one ablation sample PNG.

In [ ]:
# --- Edit these two lines, then run the cell ---
BASE_RUN = "base"
OTHER_RUN = "size_h64_b6"   # e.g. loss_pure_mse, hp_lr_5e-4, best
N_IMAGES = 4                  # must match n= used in draw_sample


def _sample_png_path(run_name, part=1):
    prefix = "p1-naivecnn" if part == 1 else "p2-unet"
    return f"output/{prefix}-{run_name}-sample.png"


def _split_sample_png(path, n_images=4):
    """Parse the 3-row grid saved by RestorationModule.draw_sample."""
    arr = np.array(Image.open(path).convert("RGB"), dtype=np.float32) / 255.0
    h, w, _ = arr.shape
    row_h, col_w = h // 3, w // n_images
    degraded, pred, clean = [], [], []
    for c in range(n_images):
        x0, x1 = c * col_w, (c + 1) * col_w
        degraded.append(arr[0:row_h, x0:x1])
        pred.append(arr[row_h : 2 * row_h, x0:x1])
        clean.append(arr[2 * row_h : 3 * row_h, x0:x1])
    return degraded, pred, clean


def _pixel_abs_error(pred, clean):
    """Mean |pred - clean| over RGB at each pixel. Shape: (H, W)."""
    return np.mean(np.abs(pred - clean), axis=2)


def compare_base_vs_other(
    base_run,
    other_run,
    part=1,
    n_images=4,
    save_dir="output",
    cmap="inferno",
    diff_cmap="RdBu_r",
):
    """
    Build: degraded | base pred | other pred | GT | base error | other error | delta error.

    Delta error is defined as:
        delta = |error(other)| - |error(base)|
    so negative values mean OTHER is better (smaller error), positive means BASE is better.
    """
    base_path = _sample_png_path(base_run, part)
    other_path = _sample_png_path(other_run, part)
    for p in (base_path, other_path):
        if not os.path.isfile(p):
            raise FileNotFoundError(f"Missing sample image: {p}\n"
                                   f"Re-run training+eval for that RUN_NAME first.")

    degraded, pred_base, clean = _split_sample_png(base_path, n_images)
    _, pred_other, _ = _split_sample_png(other_path, n_images)

    err_base = [_pixel_abs_error(pb, gt) for pb, gt in zip(pred_base, clean)]
    err_other = [_pixel_abs_error(po, gt) for po, gt in zip(pred_other, clean)]
    err_delta = [eo - eb for eo, eb in zip(err_other, err_base)]

    vmax = max(np.max(e) for e in err_base + err_other)
    vmax = max(vmax, 1e-6)  # avoid divide-by-zero if perfect reconstruction

    diff_abs_max = max(np.max(np.abs(d)) for d in err_delta)
    diff_abs_max = max(diff_abs_max, 1e-6)

    row_labels = [
        "Degraded input",
        f"Output ({base_run})",
        f"Output ({other_run})",
        "Ground truth",
        f"|error| ({base_run})",
        f"|error| ({other_run})",
        f"Delta error ({other_run} - {base_run})",
    ]
    image_rows = [
        degraded,
        pred_base,
        pred_other,
        clean,
    ]

    fig, axes = plt.subplots(
        len(row_labels), n_images,
        figsize=(2.8 * n_images, 2.5 * len(row_labels)),
    )
    if n_images == 1:
        axes = np.expand_dims(axes, axis=1)

    title = f"Part {part} - base vs {other_run}"
    fig.suptitle(title, fontsize=14, fontweight="bold", y=1.01)

    im_err = None
    im_diff = None
    for r, label in enumerate(row_labels):
        for c in range(n_images):
            ax = axes[r, c]
            if r < 4:
                ax.imshow(image_rows[r][c])
            elif r in (4, 5):
                err = err_base[c] if r == 4 else err_other[c]
                im_err = ax.imshow(err, cmap=cmap, vmin=0.0, vmax=vmax)
            else:
                im_diff = ax.imshow(
                    err_delta[c], cmap=diff_cmap,
                    vmin=-diff_abs_max, vmax=diff_abs_max,
                )
            ax.set_xticks([])
            ax.set_yticks([])
            if c == 0:
                ax.set_ylabel(label, fontsize=10)

    # Shared colorbar for absolute error rows
    cbar_err = fig.colorbar(
        im_err, ax=axes[4:6, :].ravel().tolist(),
        fraction=0.02, pad=0.02,
    )
    cbar_err.set_label("mean |pred - GT| per pixel (0-1 RGB scale)", fontsize=9)

    # Separate colorbar for delta row (diverging scale centered at 0)
    cbar_diff = fig.colorbar(
        im_diff, ax=axes[6:7, :].ravel().tolist(),
        fraction=0.02, pad=0.02,
    )
    cbar_diff.set_label("delta error: other - base (negative = other better)", fontsize=9)

    plt.tight_layout()
    plt.show()
    return None


# compare_base_vs_other(BASE_RUN, OTHER_RUN, part=1, n_images=N_IMAGES)
# Disabled to avoid auto-generating/saving the large grid image.


# Optional: single-image visualization from saved sample PNGs (no model objects needed)
def compare_vis_from_saved_samples(
    base_run,
    other_run,
    part=1,
    img_idx=0,
    crop_xywh=(80, 80, 96, 96),
    savepath=None,
):
    """
    Build a compare figure using saved files:
      output/p{part}-...-{run}-sample.png

    It parses the sample grid (degraded / pred / clean) and shows:
      - full images (degraded, base, other, clean)
      - zoomed crops
      - |base-clean| and |other-clean| heatmaps
      - |base-other| difference map
    """
    base_path = _sample_png_path(base_run, part)
    other_path = _sample_png_path(other_run, part)

    for p in (base_path, other_path):
        if not os.path.isfile(p):
            raise FileNotFoundError(f"Missing sample image: {p}")

    deg_base, pred_base, clean_base = _split_sample_png(base_path, n_images=N_IMAGES)
    deg_other, pred_other, clean_other = _split_sample_png(other_path, n_images=N_IMAGES)

    if img_idx < 0 or img_idx >= N_IMAGES:
        raise ValueError(f"img_idx must be in [0, {N_IMAGES-1}]")

    # These should be effectively identical if generated from same val sample order.
    img_deg = deg_base[img_idx]
    img_clean = clean_base[img_idx]
    img_a = pred_base[img_idx]
    img_b = pred_other[img_idx]

    err_a = np.abs(img_a - img_clean).mean(axis=2)  # |base-clean|
    err_b = np.abs(img_b - img_clean).mean(axis=2)  # |other-clean|
    err_ab = np.abs(img_a - img_b).mean(axis=2)     # |base-other|

    x, y, w, h = crop_xywh
    ys, xs = slice(y, y + h), slice(x, x + w)

    fig, ax = plt.subplots(2, 4, figsize=(16, 8))
    fig.suptitle(f"Part {part} - base vs {other_run} (image {img_idx})", fontsize=14, fontweight="bold")

    ax[0, 0].imshow(img_deg);   ax[0, 0].set_title("Degraded");          ax[0, 0].axis("off")
    ax[0, 1].imshow(img_a);     ax[0, 1].set_title(f"Base ({base_run})"); ax[0, 1].axis("off")
    ax[0, 2].imshow(img_b);     ax[0, 2].set_title(f"Other ({other_run})");ax[0, 2].axis("off")
    ax[0, 3].imshow(img_clean); ax[0, 3].set_title("Clean");             ax[0, 3].axis("off")

    # Mark same crop window on both compared predictions
    for k in [1, 2]:
        ax[0, k].add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor="yellow", linewidth=2))

    ax[1, 0].imshow(img_a[ys, xs]); ax[1, 0].set_title("Base crop");  ax[1, 0].axis("off")
    ax[1, 1].imshow(img_b[ys, xs]); ax[1, 1].set_title("Other crop"); ax[1, 1].axis("off")

    vmax_err = max(float(err_a.max()), float(err_b.max()), 1e-6)
    im2 = ax[1, 2].imshow(err_a, cmap="inferno", vmin=0.0, vmax=vmax_err)
    ax[1, 2].set_title("|Base-clean|"); ax[1, 2].axis("off")
    im3 = ax[1, 3].imshow(err_b, cmap="inferno", vmin=0.0, vmax=vmax_err)
    ax[1, 3].set_title("|Other-clean|"); ax[1, 3].axis("off")

    # Intentionally no colorbar here to avoid blocking the first comparison figure.
    plt.tight_layout()
    plt.show()

    # Separate map for direct base-vs-other difference
    fig2, ax2 = plt.subplots(1, 1, figsize=(5, 4))
    im_ab = ax2.imshow(err_ab, cmap="magma")
    ax2.set_title("|Base-Other| difference map")
    ax2.axis("off")
    cbar_ab = fig2.colorbar(im_ab, ax=ax2, fraction=0.046, pad=0.04)
    cbar_ab.set_label("mean absolute difference", fontsize=9)

    if savepath:
        fig2.savefig(savepath, dpi=150, bbox_inches="tight")
        print("saved:", savepath)
    plt.show()


# Example:
# compare_vis_from_saved_samples(
#     base_run="base", other_run="size_h64_b6", part=1,
#     img_idx=3, crop_xywh=(80, 80, 96, 96),
#     savepath="output/p1_base_vs_size_diff.png",
# )

## Part 2: UNet (45%)

The naive CNN of Part 1 processes everything at full resolution: the receptive field grows linearly with depth, and all compute is at 256x256. **UNet** ([Ronneberger et al., 2015](https://arxiv.org/abs/1505.04597)) processes features at multiple resolutions, with **skip connections** so high-frequency detail from the encoder side is fed back into the decoder side.

### Recursive structure (one suggested implementation)
A single UNet "stage" can be expressed as:

```
                       skip
input_blocks ---------------------> merge -> output_blocks
            \                     /
             down -> mid -> up
```

`mid` is **either another UNet stage or a no-op** (identity). Recursing on `mid` gives a UNet of any depth without writing the same code three times.

### Design rules

**Mandatory** (your implementation must satisfy these):
- **multi-resolution processing** — features must pass through at least one *downsample → bottleneck → upsample* path; the number of scales is up to you
- **skip connections** between matching encoder / decoder resolutions
- spatial dimensions preserved end-to-end

**Free to explore.** The recursive `UNetStage` skeleton below is one clean way to do it, but you may:
- implement the multi-scale loop iteratively or unrolled instead of recursively
- use strided conv or pooling for the down path (instead of `PixelUnshuffle`)
- use transpose conv or upsample+conv for the up path (instead of `PixelShuffle`)
- add gating / attention / extra residual connections inside stages
- vary the channel count across scales (classic UNet doubles channels each level)
- replace concat-style skips with additive skips, or vice versa

…as long as the mandatory rules above still hold.

### Ablation study

You will run the **same 5-run ablation study** as Part 1 on this architecture:

| Sub-section | What you do | Points |
|---|---|---|
| **2(a)** Implementation + Base run | Implement `UNetStage` and `UNetRestoration`; train one **base** configuration | 10% |
| **2(b)** Size ablation | Re-train with a **different size** (`HIDDEN_DIM` / `DEPTH` / `BLOCKS_PER_STAGE`) | 10% |
| **2(c)** Loss ablation | Re-train with a **different `LOSS_CFG`** | 10% |
| **2(d)** Other-hyperparameter ablation | **You pick** one hyperparameter and try at least 2 values | 10% |
| **2(e)** Best configuration | Combine the lessons above into your best UNet config | 5% |

Same workflow as Part 1: re-run the training cell with edited config and `RUN_NAME` for each ablation.

In [ ]:
# TODO: Implement one recursive UNet stage.
#       Layout (one stage):
#         x -> input_blocks -> [skip] ----------------> merge -> output_blocks -> out
#                           \                          /
#                            down -> mid -> up -------/
#       Where:
#         - input_blocks / output_blocks: blocks_per_stage residual blocks at full res
#         - down: PixelUnshuffle(2) + 1x1 Conv to keep `channels` channels (+ activation)
#         - mid: another UNetStage(depth-1) if depth > 1, else nn.Identity()
#         - up: 1x1 Conv to 4*channels + PixelShuffle(2) (+ activation)
#         - merge: 1x1 Conv from 2*channels (concat of up + skip) back to `channels`
class UNetStage(nn.Module):
    def __init__(self, channels, depth, blocks_per_stage=2):
        super().__init__()
        self.input_blocks = nn.Sequential(*[
            ResBlock(channels, kernel_size=3) for _ in range(blocks_per_stage)
        ])
        self.output_blocks = nn.Sequential(*[
            ResBlock(channels, kernel_size=3) for _ in range(blocks_per_stage)
        ])

        self.down = nn.Sequential(
            nn.PixelUnshuffle(2),        # C,H,W -> 4C,H/2,W/2
            nn.Conv2d(4 * channels, channels, kernel_size=1),
            nn.SiLU(),
        )

        self.mid = UNetStage(channels, depth - 1, blocks_per_stage) if depth > 1 else nn.Identity()

        self.up = nn.Sequential(
            nn.Conv2d(channels, 4 * channels, kernel_size=1),
            nn.SiLU(),
            nn.PixelShuffle(2),          # 4C,H/2,W/2 -> C,H,W
        )

        self.merge = nn.Conv2d(2 * channels, channels, kernel_size=1)

    def forward(self, x):
        x_in = self.input_blocks(x)
        skip = x_in

        x_low = self.down(x_in)
        x_low = self.mid(x_low)
        x_up = self.up(x_low)

        x_merge = self.merge(torch.cat([x_up, skip], dim=1))
        return self.output_blocks(x_merge)


# TODO: Implement UNetRestoration.
#       Architecture: conv_in -> UNetStage(depth) -> conv_out
class UNetRestoration(nn.Module):
    def __init__(self, in_channels=3, hidden_dim=32, depth=4, blocks_per_stage=2):
        super().__init__()
        self.conv_in = nn.Conv2d(in_channels, hidden_dim, kernel_size=3, padding=1)
        self.unet = UNetStage(hidden_dim, depth=depth, blocks_per_stage=blocks_per_stage)
        self.conv_out = nn.Conv2d(hidden_dim, in_channels, kernel_size=3, padding=1)

    def forward(self, x):
        x = self.conv_in(x)
        x = self.unet(x)
        return self.conv_out(x)


try:
    _m = UNetRestoration(hidden_dim=32, depth=4, blocks_per_stage=2)
    print("output:", _m(torch.randn(2, 3, IMG_SIZE, IMG_SIZE)).shape, "params:", count_params(_m))
except Exception as e:
    print("UNet not implemented yet:", e)

### 2(a) Implementation + Base Run (10%)

Same protocol as Part 1: edit `RUN_NAME` and the relevant config in the cell below, run training + eval, repeat for ablations 2(b)/(c)/(d) and the final best run 2(e).

In [ ]:
pl.seed_everything(42)

# === Re-run this cell for each ablation. Change RUN_NAME each time. ===
RUN_NAME = "base"

# --- Compute budget ---
MAX_STEPS = 2000
VAL_EVERY = 500
VAL_BATCHES = 50
TRAIN_BATCH_SIZE = 16
VAL_BATCH_SIZE = 64

# --- Size ---
HIDDEN_DIM = 32
DEPTH = 4                  # 4 recursive levels: 256 -> 128 -> 64 -> 32 -> 16
BLOCKS_PER_STAGE = 2

# --- Optimizer / schedule ---
LR = 2e-4
WARMUP_STEPS = 200
WEIGHT_DECAY = 1e-3

# --- Loss ---
LOSS_CFG = [{"mse": 1.0}, {"ssim": 0.1}]

train_loader, val_loader = make_loaders(TRAIN_BATCH_SIZE, VAL_BATCH_SIZE)

p2_model = UNetRestoration(
    in_channels=3, hidden_dim=HIDDEN_DIM,
    depth=DEPTH, blocks_per_stage=BLOCKS_PER_STAGE,
)
print(f"[{RUN_NAME}] Part 2 UNet params: {count_params(p2_model)/1e6:.3f} M")

p2 = RestorationModule(
    model=p2_model, loss_cfg=LOSS_CFG, lr=LR,
    total_steps=MAX_STEPS, warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY, log_period=100,
    name=f"p2-unet-{RUN_NAME}",
)

trainer = Trainer(
    max_steps=MAX_STEPS, max_epochs=-1,
    accelerator=ACCELERATOR, precision=PRECISION,
    callbacks=[LearningRateMonitor(logging_interval="step")],
    log_every_n_steps=10, gradient_clip_val=1.0,
    val_check_interval=VAL_EVERY, limit_val_batches=VAL_BATCHES,
)

trainer.fit(p2, train_loader, val_loader)

In [ ]:
# === Part 2 — Final evaluation for the run named RUN_NAME ===
val_psnr, val_ssim = final_eval(p2, val_loader)
params_M = count_params(p2.model) / 1e6
print(f"Part 2 UNet [{RUN_NAME}]  |  params: {params_M:.3f} M  "
      f"|  val PSNR: {val_psnr:.3f} dB  |  val SSIM: {val_ssim:.4f}")

deg, clean = next(iter(val_loader))
p2.draw_sample(deg, clean, n=4)
print(f"saved: output/p2-unet-{RUN_NAME}-sample.png "
      f"— inspect this for the qualitative discussion in your report.")

### 2(b) Size Ablation (10%)

Re-run the training cell above with **at least one alternative size configuration** — change one or more of `HIDDEN_DIM`, `DEPTH`, `BLOCKS_PER_STAGE`. Keep loss / optimizer / schedule **identical to the base run**. Set `RUN_NAME = "size_<tag>"`.

A tip on `DEPTH`: each level halves resolution, so `depth=4` reaches `16x16` features. Going deeper increases the receptive field at very low compute cost; going shallower keeps everything at high resolution.

### 2(c) Loss Ablation (10%)

Same instructions as 1(c): pick at least one alternative `LOSS_CFG` and re-train. Set `RUN_NAME = "loss_<tag>"`. Discuss PSNR + visual differences.

### 2(d) Hyperparameter Ablation (10%)

Same instructions as 1(d): pick **one** of `LR` / `WEIGHT_DECAY` / `WARMUP_STEPS` / `TRAIN_BATCH_SIZE` / `MAX_STEPS` / ... and try at least one alternative value. Set `RUN_NAME = "hp_<param>_<value>"`. The hyperparameter you pick here can be different from the one you ablated in Part 1 — the choice is independent.

### 2(e) Best Configuration (5%)

Same protocol as 1(e): pick the configuration you believe will perform best within the same compute budget, train it with `RUN_NAME = "best"`, and justify in 2–3 sentences.

### Part 2 Results

After running all 5 configurations, fill in the table in the cell below.

In [ ]:
# Part 2 results — fill this in after running all 5 configurations.
p2_results = [
    # ("base",         0.30, 28.9, 0.86, "h32 d4 b2, mse+0.1*ssim"),
    # ("size_d3_b3",   0.34, 29.1, 0.87, "shallower but more blocks/stage"),
    # ("loss_l1ssim",  0.30, 29.0, 0.88, "L1 + 0.1*SSIM"),
    # ("hp_lr_5e-4",   0.30, 28.7, 0.86, "higher LR"),
    # ("best",         0.55, 29.4, 0.89, "h48 d4 b3 + L1+SSIM"),
]

if p2_results:
    labels = [r[0] for r in p2_results]
    psnrs  = [r[2] for r in p2_results]
    ssims  = [r[3] for r in p2_results]

    print(f"{'run':<22} {'params (M)':>11} {'PSNR (dB)':>11} {'SSIM':>8}  note")
    print("-" * 80)
    for r in p2_results:
        print(f"{r[0]:<22} {r[1]:>11.3f} {r[2]:>11.3f} {r[3]:>8.4f}  {r[4]}")

    fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 4))
    for ax, vals, ylabel, color in [
        (axL, psnrs, "val PSNR (dB)", "darkorange"),
        (axR, ssims, "val SSIM",      "indianred"),
    ]:
        bars = ax.bar(labels, vals, color=color)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, v, f"{v:.3f}",
                    ha="center", va="bottom", fontsize=9)
        ax.set_ylabel(ylabel)
        ax.set_ylim(min(vals) - 0.5 * (max(vals) - min(vals) + 1e-3),
                    max(vals) + 0.5 * (max(vals) - min(vals) + 1e-3))
        for tick in ax.get_xticklabels():
            tick.set_rotation(20)
            tick.set_horizontalalignment("right")
    fig.suptitle("Part 2 — UNet ablation results")
    plt.tight_layout()
    plt.savefig("output/p2_results.png", dpi=120)
    plt.show()
else:
    print("Fill in p2_results above after running all 5 configurations.")

In [ ]:
@torch.no_grad()
def compare_vis(model_a, model_b, val_ds, idx=0, crop_xywh=(80, 80, 96, 96),
                title_a="A", title_b="B", savepath=None, suptitle=None):
    """Side-by-side qualitative compare for Part 1 or Part 2 (pass trained Lightning modules)."""
    model_a.eval(); model_b.eval()
    device = next(model_a.parameters()).device

    deg, clean = val_ds[idx]
    deg = deg.unsqueeze(0).to(device)
    clean = clean.unsqueeze(0).to(device)

    out_a = model_a(deg).clamp(-1, 1)
    out_b = model_b(deg).clamp(-1, 1)

    def to_img(t):
        return denormalize(t[0]).permute(1, 2, 0).detach().cpu().numpy().clip(0, 1)

    img_deg, img_clean = to_img(deg), to_img(clean)
    img_a, img_b = to_img(out_a), to_img(out_b)

    err_a = np.abs(img_a - img_clean).mean(axis=2)
    err_b = np.abs(img_b - img_clean).mean(axis=2)
    err_ab = np.abs(img_a - img_b).mean(axis=2)

    x, y, w, h = crop_xywh
    ys, xs = slice(y, y + h), slice(x, x + w)

    fig, ax = plt.subplots(2, 4, figsize=(16, 8))
    if suptitle:
        fig.suptitle(suptitle, fontsize=14, fontweight="bold")
    ax[0, 0].imshow(img_deg);   ax[0, 0].set_title("Degraded"); ax[0, 0].axis("off")
    ax[0, 1].imshow(img_a);     ax[0, 1].set_title(title_a);   ax[0, 1].axis("off")
    ax[0, 2].imshow(img_b);     ax[0, 2].set_title(title_b);   ax[0, 2].axis("off")
    ax[0, 3].imshow(img_clean); ax[0, 3].set_title("Clean");    ax[0, 3].axis("off")

    for k in [1, 2]:
        ax[0, k].add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor="yellow", linewidth=2))

    vmax_err = max(float(err_a.max()), float(err_b.max()), 1e-6)
    ax[1, 0].imshow(img_a[ys, xs]); ax[1, 0].set_title(f"{title_a} crop"); ax[1, 0].axis("off")
    ax[1, 1].imshow(img_b[ys, xs]); ax[1, 1].set_title(f"{title_b} crop"); ax[1, 1].axis("off")
    ax[1, 2].imshow(err_a, cmap="inferno", vmin=0.0, vmax=vmax_err)
    ax[1, 2].set_title(f"|{title_a}-clean|"); ax[1, 2].axis("off")
    ax[1, 3].imshow(err_b, cmap="inferno", vmin=0.0, vmax=vmax_err)
    ax[1, 3].set_title(f"|{title_b}-clean|"); ax[1, 3].axis("off")
    plt.tight_layout()
    plt.show()

    fig2, ax2 = plt.subplots(1, 1, figsize=(5, 4))
    im_ab = ax2.imshow(err_ab, cmap="magma")
    ax2.set_title("|A-B| difference map")
    ax2.axis("off")
    fig2.colorbar(im_ab, ax=ax2, fraction=0.046, pad=0.04)
    if savepath:
        fig2.savefig(savepath, dpi=150, bbox_inches="tight")
        print("saved:", savepath)
    plt.show()


# --- Part 2: model-based (run right after eval while both modules are in memory) ---
# compare_vis(
#     p2_base, p2_best, val_ds, idx=3,
#     title_a="Base", title_b="Best",
#     suptitle="Part 2 UNet - base vs best",
#     savepath="output/p2_base_vs_best_diff.png",
# )

# --- Part 2: from saved sample PNGs (no need to keep old checkpoints loaded) ---
# compare_vis_from_saved_samples(
#     base_run="base", other_run="best", part=2,
#     img_idx=3, crop_xywh=(80, 80, 96, 96),
#     savepath="output/p2_base_vs_best_diff.png",
# )

### Part 1 vs Part 2 — Final Comparison

The cell below compares your **best** Part 1 NaiveCNN (`p1`) against your **best** Part 2 UNet (`p2`) on the same val images. Run this *after* you have re-trained both with `RUN_NAME = "best"`. The figure is saved to `output/p1_vs_p2.png` and used in the report.

In [ ]:
@torch.no_grad()
def compare_models(modules, names, val_ds, n_images=4, savepath="output/p1_vs_p2.png"):
    """modules: list of LightningModule (already trained), names: list of str."""
    device = next(modules[0].parameters()).device
    deg, clean = zip(*[val_ds[i] for i in range(n_images)])
    deg = torch.stack(deg).to(device)
    clean = torch.stack(clean).to(device)

    rows = [deg.cpu()]
    row_labels = ["Degraded"]
    for m, name in zip(modules, names):
        m.eval()
        out = m(deg).clamp(-1, 1).cpu()
        rows.append(out)
        row_labels.append(name)
    rows.append(clean.cpu())
    row_labels.append("Clean")

    fig, axes = plt.subplots(len(rows), n_images, figsize=(3 * n_images, 3 * len(rows)))
    for r, (row, label) in enumerate(zip(rows, row_labels)):
        for c in range(n_images):
            ax = axes[r, c] if n_images > 1 else axes[r]
            ax.imshow(denormalize(row[c]).permute(1, 2, 0).numpy().clip(0, 1))
            ax.axis("off")
            if c == 0:
                ax.set_ylabel(label, fontsize=12)
                ax.axis("on")
                ax.set_xticks([])
                ax.set_yticks([])
    plt.tight_layout()
    plt.savefig(savepath, dpi=120)
    plt.show()


# Run after both p1 and p2 hold the "best" run.
compare_models([p1, p2], ["P1 NaiveCNN (best)", "P2 UNet (best)"], val_ds, n_images=4)

## Report (10%)

Write your report below using markdown. **Every section must combine quantitative numbers (PSNR + SSIM) with a qualitative discussion** — embed or reference the saved sample images (`output/*-sample.png`) and describe what you actually see, not just the metric.

**I. Part 1 — NaiveCNN ablation summary (3%)**
Refer to your `p1_results` table and the saved sample images. For each ablation axis (size / loss / hp), report:
- the *direction* and *magnitude* of the change in **both** PSNR and SSIM (e.g. "doubling `hidden_dim` gained ~0.5 dB PSNR and +0.01 SSIM but quadrupled params");
- a *qualitative* description from the sample images — what kind of artifact got better or worse? Block edges? Color bleed? Texture detail? Smoothness?
- end with a 2–3 sentence justification of the configuration you picked for `1(e) best`.

**II. Part 2 — UNet ablation summary (3%)**
Same structure as I, for the UNet runs in `p2_results`.

**III. Part 1 vs Part 2 — architecture comparison (3%)**
Using your **best** Part 1 and Part 2 runs (and the figure in `output/p1_vs_p2.png`), report:
- val PSNR and val SSIM for each
- which architecture wins on each metric, by how much, at what parameter cost
- **what the UNet visibly fixes that the flat residual stack misses (or doesn't)** — point to specific spots in the side-by-side images. Metric agreement vs. visual disagreement is itself worth discussing.

**IV. Reflection (1%)**
Of the two architectures, which would you actually deploy for compression-artifact + noise restoration on phone photos, and why? Consider PSNR, SSIM, parameter count, inference cost, *and* the qualitative behavior you observed.

> **Note on metrics vs. visual quality.** PSNR and SSIM are convenient but neither perfectly predicts perceived quality. A higher-PSNR model can look blurrier; a higher-SSIM model can drop fine texture. The HW grades on the depth of your discussion, not on whether you hit the highest number — pick honest comparisons backed by both the table and the images.